# 12 — LLM Token Guard (`core/llm_guard.py`)
Daily token budget enforcement via Redis counter.

- **`check_and_record_tokens(redis, tokens)`** — increments counter, returns `True` (OK) or `False` (budget exceeded → caller raises 429)
- **`get_daily_usage(redis)`** — returns current usage stats for `/agents/status` dashboard  
- **`estimate_tokens(text)`** — rough estimate: `len(text)//4 + 500`

**Fail-open**: Redis error → always returns `True` (never blocks traffic on infra failure)


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'codebase', 'codebase', 'src'))
os.makedirs("logs", exist_ok=True)

## 1. estimate_tokens — Rough Token Counter

In [ ]:
from core.llm_guard import estimate_tokens

texts = [
    "What is GRR?",
    "What is the gross retention rate for the retention metric in analytics.retention_metrics?",
    "A" * 400,  # 400 chars ≈ 100 tokens + 500 overhead
]
for t in texts:
    est = estimate_tokens(t)
    print(f"  chars={len(t):5d} | estimated_tokens={est}  | text='{t[:50]}'")

## 2. check_and_record_tokens — No Redis (Dev Mode)

In [ ]:
from core.llm_guard import check_and_record_tokens

# No Redis client → fail-open, always allowed
result = check_and_record_tokens(None, tokens_used=1000)
print("No Redis client → allowed:", result)  # always True

## 3. get_daily_usage — No Redis

In [ ]:
from core.llm_guard import get_daily_usage, DAILY_TOKEN_LIMIT

usage = get_daily_usage(None)  # no Redis
print("Daily usage stats (no Redis):")
for k, v in usage.items():
    print(f"  {k:<15}: {v}")
print(f"\nDAILY_TOKEN_LIMIT: {DAILY_TOKEN_LIMIT:,}")

## 4. Simulate Redis with a Dict (mock)

In [ ]:
# Simulate Redis behavior using a dict-based mock
from datetime import date

class MockRedis:
    def __init__(self):
        self._data = {}
    def incrby(self, key, amount):
        self._data[key] = self._data.get(key, 0) + amount
        return self._data[key]
    def expire(self, key, seconds):
        pass  # no-op
    def get(self, key):
        val = self._data.get(key)
        return str(val) if val is not None else None

redis = MockRedis()

# Record some usage
calls = [500, 1000, 2500, 750]
for tokens in calls:
    allowed = check_and_record_tokens(redis, tokens_used=tokens)
    usage = get_daily_usage(redis)
    print(f"  +{tokens:4d} tokens → allowed={allowed} | total={usage['tokens_used']:,} ({usage['pct']}%)")

## 5. Simulate Budget Exceeded

In [ ]:
# Manually push counter near the limit
from core.llm_guard import DAILY_TOKEN_LIMIT
redis2 = MockRedis()
key = f"token_usage:{date.today().isoformat()}"

# Manually set near limit
redis2._data[key] = DAILY_TOKEN_LIMIT - 100  # 100 tokens remaining

# First call: just within budget
result1 = check_and_record_tokens(redis2, tokens_used=50)
print(f"Used 50 tokens: allowed={result1} | remaining≈50")

# Second call: exceeds limit
result2 = check_and_record_tokens(redis2, tokens_used=200)
print(f"Used 200 tokens: allowed={result2}  ← should be False (budget exceeded)")

usage = get_daily_usage(redis2)
print(f"\nFinal usage: {usage['tokens_used']:,} / {usage['limit']:,} ({usage['pct']}%)")

## 6. Fail-Open: Redis Error

In [ ]:
class BrokenRedis:
    def incrby(self, key, amount):
        raise ConnectionError("Redis connection refused")
    def expire(self, key, seconds):
        raise ConnectionError("Redis connection refused")
    def get(self, key):
        raise ConnectionError("Redis connection refused")

broken = BrokenRedis()
result = check_and_record_tokens(broken, tokens_used=999)
print("Broken Redis → allowed:", result)  # True — fail open, never blocks traffic

usage = get_daily_usage(broken)
print("Broken Redis usage:", usage)